In [16]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from transformers import set_seed, EsmModel, AutoTokenizer, EsmForMaskedLM
from functools import partial
import pandas as pd
import numpy as np

In [2]:
def get_esm(type, masked_lm=False):
    if masked_lm:
        model_class = EsmForMaskedLM
    else:
        model_class = EsmModel

    if type == '3B':
        model, tokenizer = model_class.from_pretrained('facebook/esm2_t36_3B_UR50D'), AutoTokenizer.from_pretrained('facebook/esm2_t36_3B_UR50D')
    elif type == '15B':
        model, tokenizer = model_class.from_pretrained('facebook/esm2_t48_15B_UR50D'), AutoTokenizer.from_pretrained('facebook/esm2_t48_15B_UR50D')
    elif type == '35M':
        model, tokenizer = model_class.from_pretrained('facebook/esm2_t12_35M_UR50D'), AutoTokenizer.from_pretrained('facebook/esm2_t12_35M_UR50D')
    else:
        model, tokenizer = model_class.from_pretrained('facebook/esm2_t33_650M_UR50D'), AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
    return model, tokenizer


In [42]:
def labeling_fn(sequence, sites, residues={'S', 'T', 'Y'}, ignore_index=-1):
    res = np.zeros(len(sequence), dtype=np.int32) + ignore_index
    mask = [s in residues for s in sequence] # Only relevant prots are not ignored
    res[mask] = 0
    valid_sites = [i for i in sites if sequence[i] in residues]
    res[valid_sites] = 1

    return res

def prep_batch(data, tokenizer, ignore_label=-1, kinases=False):
    """
    Collate function for a dataloader. "data" is a list of inputs.

    Return a dictionary with keys [input_ids, labels, batch_lens, indices]
    """
    # Indices are for the protein dataframe
    if kinases:
        indices, sequences, labels, kinase_labels = zip(*data)
    else:
        indices, sequences, labels = zip(*data)
        
    batch = tokenizer(sequences, padding='longest', return_tensors="pt")
    sequence_length = batch["input_ids"].shape[1]

    # Pad the labels correctly
    batch['labels'] = np.array([[ignore_label] + list(label) + [ignore_label] * (sequence_length - len(label) - 1) for label in labels])
    batch['labels'] = torch.as_tensor(batch['labels'], dtype=torch.float32)
    batch['batch_lens'] = torch.as_tensor(np.array([len(x) for x in labels]))
    batch['indices'] = torch.as_tensor(np.array(indices, dtype=np.int32))
    if kinases:
        batch['kinase_labels'] = torch.as_tensor(np.asarray([val for lst in kinase_labels for val in lst]), dtype=torch.float32)

    return batch

class ProteinDataset(Dataset):
    def __init__(self, data : pd.DataFrame, kinase_labels=False) -> None:
        self.data = data
        self.kinase_labels = kinase_labels

    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, index : int):
        row = self.data.iloc[index]
        if self.kinase_labels:
            return  index, row.sequence, row.label, row.kinase_labels
        
        return index, row.sequence, row.label


In [30]:
prot_info_path = '../data/dbptm/dbptm_info_split_prots_only.json'
prot_info = pd.read_json(prot_info_path)
residues = {'S', 'T', 'Y'}
prot_info.loc[:, 'sites'] = prot_info.apply(lambda x: list(set([int(site) - 1 for site in x['sites'] if x['sequence'][int(site) - 1] in residues])), axis=1)
prot_info['label'] = [labeling_fn(sequence, site_list, residues=residues, ignore_index=-1) for sequence, site_list in zip(prot_info.sequence, prot_info.sites)]

In [43]:
dataset = ProteinDataset(prot_info)
dataset[0]

(0,
 'MRIQKQQYTISSNSRINLLGILVLNVVCGKSSIFFSHPQRLGKLGGSSLGSTGPFQTLSINFCIGCFLFNSNHFDLLFSLPSSSSILSMSVLEKFCSCIDSVTRCCPSQSLETPGSVASHVVLALSSKCTPIQFNAKWSISHKSNTG',
 array([-1, -1, -1, -1, -1, -1, -1,  0,  0, -1,  0,  0, -1,  0, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0, -1, -1,
        -1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0,  0, -1, -1,  0,
         0, -1, -1, -1, -1,  0, -1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1,  0, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1,  0,  0,  0,  0,
        -1, -1,  0, -1,  0, -1, -1, -1, -1, -1, -1,  0, -1, -1, -1,  0, -1,
         0, -1, -1, -1, -1,  0, -1,  0, -1, -1,  1, -1, -1,  0, -1, -1,  1,
        -1, -1, -1, -1, -1, -1,  0,  0, -1, -1,  0, -1, -1, -1, -1, -1, -1,
        -1, -1,  0, -1,  0, -1, -1,  0, -1,  0, -1], dtype=int32))

In [48]:
esm, tokenizer = get_esm('650M')
device='cuda:0' if torch.cuda.is_available() else 'cpu'
esm.to(device)

Loading weights: 100%|██████████| 534/534 [00:00<00:00, 9654.92it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
        )
        (output): Es

In [55]:
loader = DataLoader(dataset, batch_size=1, collate_fn=partial(prep_batch, tokenizer=tokenizer))

In [56]:
import os

out_folder = '../data/esm_embeds'
os.makedirs(out_folder,exist_ok=True)

for batch in loader:
    with torch.no_grad():
        out = esm(**batch.to(device))
        embeds = out[0]

        for i, df_idx in enumerate(batch['indices']):
            res_embeds = embeds[i]
            np.savez_compressed(f'{out_folder}/{dataset[df_idx].id}.npz', a=res_embeds.cpu.numpy())

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 2.62 MiB is free. Including non-PyTorch memory, this process has 7.41 GiB memory in use. Of the allocated memory 7.17 GiB is allocated by PyTorch, and 116.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)